# Geosteering AI — Sprint O1: Validação GPU (pytest -m gpu)

**Template:** `validate_sprint_o1_gpu_tests.ipynb` | **Sprint:** O1 (`sprint-o1-quick-wins`)

Executa **todos os testes `@pytest.mark.gpu`** do simulador JAX (≈96 testes GPU-only)
para validar que as otimizações do Sprint O1 não introduziram regressões e que
os Quick Wins #1-#9 estão ativos e funcionando corretamente.

## O que este notebook valida

| § | Seção | Testes |
|:-:|:------|:-------|
| 3 | Detecção GPU + info de device | — |
| 4 | Verificação das otimizações O1 ativas | — |
| 5 | Suite pytest `-m gpu` (todos JAX) | ≈96 testes |
| 6 | Paridade Fortran `<1e-12` (CPU) | 10 modelos |
| 7 | Gate de performance T1.6 (throughput) | 1 teste |
| 8 | Export resultados → Google Drive | — |
| 9 | Decisão Go / No-Go para merge em `main` | — |

## Pré-requisitos

- **Colab Pro+** com runtime GPU (T4 ou A100)
- **Google Drive** montado (para clone e export)
- **Tempo estimado:** 15–30 min (inclui cold-start XLA e compile JIT)

## Quick Wins verificados (Sprint O1)

| # | Otimização | Verificação |
|:-:|:-----------|:------------|
| 1 | `_layer_at_depth_vec` (NumPy vetorizado) | Importação sem erro |
| 2 | `lax.scan(unroll=K)` substitui `fori_loop` | `cfg.unroll_layer_loop=True` default |
| 3 | XLA persistent compilation cache | `JAX_COMPILATION_CACHE_DIR` configurado |
| 4 | XLA flag `latency_hiding_scheduler` | `XLA_FLAGS` contém a flag (**triton_softmax removida — causa FATAL em JAX 0.7+**) |
| 5 | `jax.config.jax_compilation_cache_dir` | API moderna ativa |
| 6 | `get_hankel_filter_jax` LRU cache | Cache hit na 2ª chamada |
| 7 | Sync GPU→CPU deferido ao caller | `np.asarray` final (não `block_until_ready`) |
| 8 | `jax.tree.map` consolida HtoD transfers | Wired em batched + vmap_real |
| 9 | `_sanitize_profile_batch` vetorizado | `np.cumsum` axis=1 |


In [ ]:
# ── Célula 1: Env vars XLA — DEVE rodar ANTES de qualquer `import jax` ─────
# Incluindo antes do pip install (que importa jax internamente).
# Reproduz exatamente _setup_xla_environment() em _jax/__init__.py.
import os

# Quick Win #3 — Persistent XLA Compilation Cache
JAX_CACHE_DIR = "/tmp/jax_compilation_cache_geosteering"
os.makedirs(JAX_CACHE_DIR, exist_ok=True)
os.environ.setdefault("JAX_COMPILATION_CACHE_DIR", JAX_CACHE_DIR)

# Quick Win #4 — XLA flags A100/T4
# NOTA: xla_gpu_enable_triton_softmax_fusion foi removida em JAX >= 0.5.x / XLA 0.7+.
# Setar flag desconhecida causa FATAL em parse_flags_from_env.cc — mata o kernel.
# Mantemos apenas latency_hiding_scheduler, que é estável em JAX 0.4-0.7.
_existing = os.environ.get("XLA_FLAGS", "")
_flags = [
    "--xla_gpu_enable_latency_hiding_scheduler=true",
]
os.environ["XLA_FLAGS"] = " ".join(
    [_existing] + [f for f in _flags if f not in _existing]
).strip()

# VRAM allocation — BFC allocator (evita pre-alocação eager de 75%)
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.85")

print("[OK] Env vars XLA configuradas antes do import jax")
print(f"     JAX_COMPILATION_CACHE_DIR = {os.environ['JAX_COMPILATION_CACHE_DIR']}")
print(f"     XLA_FLAGS = {os.environ['XLA_FLAGS']}")
print(f"     XLA_PYTHON_CLIENT_PREALLOCATE = {os.environ['XLA_PYTHON_CLIENT_PREALLOCATE']}")
print(f"     XLA_PYTHON_CLIENT_MEM_FRACTION = {os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION']}")


## § 1 Configurações da Sprint

In [ ]:
# ── Célula 3: Configurações ──────────────────────────────────────────────────
GIT_BRANCH  = "feat/sprint-o1-quick-wins"  # Branch Sprint O1
GIT_TAG     = "v2.43"                       # Tag de release (atualizar após merge)
REPO_URL    = "https://github.com/daniel-guitarplayer-8/geosteering-ai.git"
REPO_DIR    = "/content/geosteering_ai_repo"
DRIVE_OUT   = "/content/drive/MyDrive/geosteering_ai_results/sprint_o1"

# Arquivos de teste GPU-only
GPU_TEST_FILES = [
    "tests/test_simulation_jax_sprint10_parity.py",    # 8 testes — Sprint 10 Phase 2
    "tests/test_simulation_jax_sprint10_wired.py",     # 5 testes — Sprint 10 wired
    "tests/test_simulation_jax_sprint12.py",           # 23 testes — Sprint 12 vmap_real
    "tests/test_simulation_jax_sprint13.py",           # 4 testes — Sprint 13
    "tests/test_simulation_jax_dipoles_native.py",     # 15 testes — HMD native
    "tests/test_simulation_jax_vmd_native.py",         # 8 testes — VMD native
    "tests/test_simulation_jax_native_e2e.py",         # 8 testes — end-to-end
    "tests/test_simulation_jax_multi.py",              # 5 testes — multi_forward
    "tests/test_simulation_jax_performance.py",        # 16 testes — performance
    "tests/test_simulation_jax_batched_api.py",        # 3 gpu-marked — batched API
    "tests/test_simulation_jax_perf_baseline.py",      # 1 gpu-marked — gate T1.6
]

print(f"Branch : {GIT_BRANCH}")
print(f"Tag    : {GIT_TAG}")
print(f"Repo   : {REPO_URL}")
print(f"Arquivos de teste GPU: {len(GPU_TEST_FILES)} ficheiros")


## § 2 Setup do Ambiente

Monta Drive, clona repositório, instala dependências.
**Importante:** a instalação do JAX `cuda12` deve vir APÓS a célula 1 (env vars).

In [ ]:
# ── Célula 5: Setup ─────────────────────────────────────────────────────────
import subprocess, sys

# 1. Montar Google Drive
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("[INFO] Não está em Colab — execução local (sem Drive)")

# 2. Criar diretório de output no Drive
if IN_COLAB:
    import os
    os.makedirs(DRIVE_OUT, exist_ok=True)
    print(f"[OK] Drive output dir: {DRIVE_OUT}")

# 3. Clonar repositório
import shutil
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

print(f"Clonando {GIT_BRANCH} ...")
result = subprocess.run(
    ["git", "clone", "--depth=1", "--branch", GIT_BRANCH, REPO_URL, REPO_DIR],
    capture_output=True, text=True,
)
if result.returncode != 0:
    # Fallback: clone main e faz checkout da branch
    print(f"[WARN] Branch não encontrada ({result.stderr[:100]}); clonando main ...")
    subprocess.run(
        ["git", "clone", "--depth=1", REPO_URL, REPO_DIR],
        check=True, capture_output=True,
    )
    subprocess.run(
        ["git", "-C", REPO_DIR, "fetch", "origin", GIT_BRANCH],
        capture_output=True,
    )
    subprocess.run(
        ["git", "-C", REPO_DIR, "checkout", GIT_BRANCH],
        capture_output=True,
    )

commit = subprocess.run(
    ["git", "-C", REPO_DIR, "log", "--oneline", "-1"],
    capture_output=True, text=True,
).stdout.strip()
print(f"[OK] Repositório clonado: {commit}")

# 4. Adicionar ao sys.path
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# 5. Instalar JAX CUDA + dependências
print("\nInstalando jax[cuda12_pip] (aguarde ~2 min) ...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "jax[cuda12_pip]", "-f", "https://storage.googleapis.com/jax-releases/jax_cuda_releases.html"],
    check=True,
)
print("Instalando geosteering_ai + dependências ...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[dev]"],
    check=True,
)
print("[OK] Instalação concluída")


## § 3 Detecção GPU e Info de Device

Verifica backend JAX, device(s) disponíveis e versão CUDA/cuDNN.
**Critério de aceitação:** `jax.default_backend()` em `{"gpu", "cuda"}` e
device contém `"cuda"` ou `"gpu"` (case-insensitive).

In [ ]:
# ── Célula 7: Detecção GPU ──────────────────────────────────────────────────
import jax
import jax.numpy as jnp

# Habilita float64 (critical para paridade Fortran <1e-12)
jax.config.update("jax_enable_x64", True)

backend = jax.default_backend()
devices = jax.devices()

print(f"JAX version   : {jax.__version__}")
print(f"Backend       : {backend}")
print(f"Devices ({len(devices)}):")
for d in devices:
    print(f"  {d}")

# Verificação GPU
IS_GPU = backend in ("gpu", "cuda") or any(
    "cuda" in str(d).lower() or "gpu" in str(d).lower() for d in devices
)

if IS_GPU:
    print("\n[PASS] GPU detectada — todos os testes @pytest.mark.gpu serão executados")
    # Info adicional via nvidia-smi
    try:
        import subprocess as _sp
        smi = _sp.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version,compute_cap",
                       "--format=csv,noheader"], capture_output=True, text=True)
        if smi.returncode == 0:
            print(f"[GPU] {smi.stdout.strip()}")
    except FileNotFoundError:
        pass
else:
    print("\n[WARN] GPU NÃO detectada — testes @pytest.mark.gpu serão SKIPPED")
    print("       Verifique: Runtime → Change runtime type → GPU (T4 ou A100)")

# Smoke test: operação simples na GPU
x = jnp.ones((4, 4), dtype=jnp.float64)
y = jnp.dot(x, x)
print(f"\n[OK] Smoke test jnp.dot float64: shape={y.shape}, sum={float(jnp.sum(y)):.1f} (esperado 64.0)")

assert float(jnp.sum(y)) == 64.0, "Smoke test falhou!"


## § 4 Verificação das Otimizações Sprint O1

Confirma que os 9 Quick Wins estão ativos e configurados corretamente,
**antes** de rodar a suite de testes.

In [ ]:
# ── Célula 9: Verificação das otimizações O1 ─────────────────────────────────
import os, time
import numpy as np

checks = []

# ── #3 XLA persistent cache ───────────────────────────────────────────────────
cache_dir = os.environ.get("JAX_COMPILATION_CACHE_DIR", "")
checks.append(("#3 XLA cache dir setado", bool(cache_dir), cache_dir or "NÃO SETADO"))

# ── #4 XLA flags ──────────────────────────────────────────────────────────────
# Apenas latency_hiding_scheduler — triton_softmax_fusion removida em JAX 0.5+/XLA 0.7+
xla_flags = os.environ.get("XLA_FLAGS", "")
has_latency = "xla_gpu_enable_latency_hiding_scheduler=true" in xla_flags
checks.append(("#4 XLA flag latency_hiding_scheduler", has_latency, xla_flags[:80] or "não setado"))

# ── #4 VRAM config ───────────────────────────────────────────────────────────
no_prealloc = os.environ.get("XLA_PYTHON_CLIENT_PREALLOCATE", "") == "false"
mem_frac    = os.environ.get("XLA_PYTHON_CLIENT_MEM_FRACTION", "")
checks.append(("#4 PREALLOCATE=false", no_prealloc, os.environ.get("XLA_PYTHON_CLIENT_PREALLOCATE", "não setado")))
checks.append(("#4 MEM_FRACTION=0.85", mem_frac == "0.85", mem_frac or "não setado"))

# ── #2 lax.scan unroll flag em SimulationConfig ───────────────────────────────
try:
    from geosteering_ai.simulation.config import SimulationConfig
    cfg = SimulationConfig()
    unroll_active = getattr(cfg, "unroll_layer_loop", None)
    checks.append(("#2 unroll_layer_loop default=True", unroll_active is True, str(unroll_active)))
except Exception as e:
    checks.append(("#2 unroll_layer_loop", False, str(e)))

# ── #6 Hankel LRU cache ───────────────────────────────────────────────────────
try:
    from geosteering_ai.simulation._jax.kernel import get_hankel_filter_jax
    # Chama 2x e mede: 1ª chama FilterLoader, 2ª usa cache
    t0 = time.perf_counter()
    _ = get_hankel_filter_jax("werthmuller_201pt")
    t1 = time.perf_counter()
    _ = get_hankel_filter_jax("werthmuller_201pt")  # deve ser muito mais rápido
    t2 = time.perf_counter()
    first_ms  = (t1 - t0) * 1000
    second_ms = (t2 - t1) * 1000
    cache_speedup = first_ms / max(second_ms, 1e-6)
    checks.append((f"#6 Hankel LRU (speedup 2ª call: {cache_speedup:.0f}×)",
                   cache_speedup > 10,
                   f"{first_ms:.1f}ms → {second_ms:.3f}ms"))
except Exception as e:
    checks.append(("#6 Hankel LRU cache", False, str(e)))

# ── #1 _layer_at_depth_vec (NumPy vetorizado) ────────────────────────────────
try:
    from geosteering_ai.simulation._jax.multi_forward import _layer_at_depth_vec
    prof = np.array([0.0, 1.0, 2.0])
    idx = _layer_at_depth_vec(4, 0.5, prof)
    checks.append(("#1 _layer_at_depth_vec importado", True, f"test idx={idx}"))
except ImportError as e:
    checks.append(("#1 _layer_at_depth_vec", False, str(e)))
except Exception as e:
    checks.append(("#1 _layer_at_depth_vec importado", True, f"importado (teste de call erro: {e})"))

# ── #9 _sanitize_profile_batch ───────────────────────────────────────────────
try:
    from geosteering_ai.simulation._jax.multi_forward import _sanitize_profile_batch
    checks.append(("#9 _sanitize_profile_batch importado", True, "OK"))
except ImportError as e:
    checks.append(("#9 _sanitize_profile_batch", False, str(e)))

# ── Relatório ─────────────────────────────────────────────────────────────────
print("Sprint O1 — Verificação de Otimizações")
print("=" * 60)
n_pass = 0
for label, ok, detail in checks:
    status = "[PASS]" if ok else "[FAIL]"
    if ok:
        n_pass += 1
    print(f"{status} {label}")
    if not ok or "speedup" in label:
        print(f"       {detail}")

print("=" * 60)
print(f"{n_pass}/{len(checks)} verificações passaram")
if n_pass < len(checks):
    print("[WARN] Algumas otimizações não confirmadas — verifique os FAIL acima")
else:
    print("[OK] Todas as otimizações Sprint O1 confirmadas")


## § 5 Suite de Testes JAX GPU (`pytest -m gpu`)

Executa todos os arquivos de teste com testes `@pytest.mark.gpu` sequencialmente,
com timeout de 10 min por arquivo. Salva saída completa e contabiliza resultados.

**Critério de gate:** `0 FAILED` em todos os arquivos.

In [ ]:
# ── Célula 11: pytest -m gpu ────────────────────────────────────────────────
import subprocess, sys, time, re, os

GPU_RESULTS = []  # acumula resultados por arquivo

print(f"Rodando pytest -m gpu em {len(GPU_TEST_FILES)} arquivos")
print("=" * 70)

for test_file in GPU_TEST_FILES:
    abs_path = os.path.join(REPO_DIR, test_file)
    if not os.path.exists(abs_path):
        GPU_RESULTS.append({
            "file": test_file,
            "passed": 0, "failed": 0, "skipped": 0, "error": "arquivo não encontrado",
            "elapsed_s": 0.0, "output": "",
        })
        print(f"[SKIP] {os.path.basename(test_file)} — arquivo não encontrado")
        continue

    t0 = time.perf_counter()
    try:
        result = subprocess.run(
            [sys.executable, "-m", "pytest",
             "-m", "gpu",
             "-v", "--tb=short",
             "--no-header",
             "-p", "no:warnings",
             abs_path],
            capture_output=True, text=True,
            timeout=600,
            cwd=REPO_DIR,
        )
    except subprocess.TimeoutExpired:
        elapsed = time.perf_counter() - t0
        GPU_RESULTS.append({
            "file": test_file,
            "passed": 0, "failed": 0, "skipped": 0, "error": "TIMEOUT >600s",
            "elapsed_s": elapsed, "output": "",
        })
        print(f"[TIMEOUT] {os.path.basename(test_file)} — >600s")
        continue

    elapsed = time.perf_counter() - t0
    stdout = result.stdout + result.stderr

    # Parse summary line: "X passed, Y failed, Z skipped in Ns"
    m_pass    = re.search(r"(\d+) passed",   stdout)
    m_fail    = re.search(r"(\d+) failed",   stdout)
    m_skip    = re.search(r"(\d+) skipped",  stdout)
    m_error   = re.search(r"(\d+) error",    stdout)

    n_passed  = int(m_pass.group(1))  if m_pass  else 0
    n_failed  = int(m_fail.group(1))  if m_fail  else 0
    n_skipped = int(m_skip.group(1))  if m_skip  else 0
    n_errors  = int(m_error.group(1)) if m_error else 0

    # Extrair linhas FAILED para display rápido
    failed_lines = [ln for ln in stdout.split("\n") if ln.strip().startswith("FAILED")]

    status = "PASS" if (n_failed == 0 and n_errors == 0 and result.returncode == 0) else "FAIL"
    GPU_RESULTS.append({
        "file": test_file,
        "passed": n_passed, "failed": n_failed, "skipped": n_skipped,
        "errors": n_errors, "status": status,
        "elapsed_s": elapsed, "output": stdout,
        "failed_lines": failed_lines,
    })

    tag = "[PASS]" if status == "PASS" else "[FAIL]"
    print(f"{tag} {os.path.basename(test_file):55s} "
          f"{n_passed:3d}P {n_skipped:3d}S {n_failed:3d}F  {elapsed:5.1f}s")
    if failed_lines:
        for fl in failed_lines[:5]:
            print(f"       {fl.strip()}")

# ── Sumário ──────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
total_passed  = sum(r.get("passed",  0) for r in GPU_RESULTS)
total_failed  = sum(r.get("failed",  0) for r in GPU_RESULTS)
total_skipped = sum(r.get("skipped", 0) for r in GPU_RESULTS)
total_time    = sum(r.get("elapsed_s", 0) for r in GPU_RESULTS)

print(f"TOTAL: {total_passed} PASSED | {total_failed} FAILED | {total_skipped} SKIPPED | {total_time:.0f}s")

GPU_GATE_OK = total_failed == 0
if GPU_GATE_OK:
    print("[GATE PASS] 0 FAILED — Suite GPU aprovada")
else:
    print(f"[GATE FAIL] {total_failed} FAILED — Regressão detectada!")


## § 6 Paridade Fortran `<1e-12` (CPU)

Executa os 10 modelos canônicos de paridade Fortran. Estes testes
**não requerem GPU** (rodam via Numba CPU) mas são obrigatórios
para confirmar que nenhuma mudança do Sprint O1 quebrou a paridade numérica.

In [ ]:
# ── Célula 13: Paridade Fortran <1e-12 ──────────────────────────────────────
import subprocess, sys, time, re, os

PARITY_FILE = os.path.join(REPO_DIR, "tests/test_simulation_jax_fortran_parity.py")

print("Rodando paridade Fortran <1e-12 ...")
t0 = time.perf_counter()

result_parity = subprocess.run(
    [sys.executable, "-m", "pytest",
     "-v", "--tb=short", "--no-header", "-p", "no:warnings",
     PARITY_FILE],
    capture_output=True, text=True, timeout=300, cwd=REPO_DIR,
)

elapsed_parity = time.perf_counter() - t0
stdout_parity = result_parity.stdout + result_parity.stderr

m_pass  = re.search(r"(\d+) passed",  stdout_parity)
m_fail  = re.search(r"(\d+) failed",  stdout_parity)
n_passed = int(m_pass.group(1)) if m_pass else 0
n_failed = int(m_fail.group(1)) if m_fail else 0

PARITY_GATE_OK = n_failed == 0 and result_parity.returncode == 0

print(stdout_parity[-3000:])  # últimas 3000 chars (summary + failures)
print(f"\n{'[GATE PASS]' if PARITY_GATE_OK else '[GATE FAIL]'} "
      f"Paridade Fortran: {n_passed} PASSED, {n_failed} FAILED  ({elapsed_parity:.1f}s)")


## § 7 Gate de Performance T1.6

Executa o teste de baseline de performance (`test_simulation_jax_perf_baseline.py`).
Este é o gate T1.6 do Sprint O0: throughput JAX GPU ≥ throughput Numba CPU (ou
target específico configurado no teste).

**Nota:** O cold-start inclui compilação XLA (~30–180s). Execuções subsequentes
usam o cache persistente e são significativamente mais rápidas.

In [ ]:
# ── Célula 15: Gate T1.6 — Performance ─────────────────────────────────────
import subprocess, sys, time, re, os

PERF_FILE = os.path.join(REPO_DIR, "tests/test_simulation_jax_perf_baseline.py")

print("Rodando gate T1.6 (performance baseline) ...")
print("[INFO] Cold-start inclui compilação XLA — pode levar 2–5 min")

t0 = time.perf_counter()
result_perf = subprocess.run(
    [sys.executable, "-m", "pytest",
     "-m", "gpu",
     "-v", "--tb=long", "--no-header", "-p", "no:warnings",
     "-s",  # permite print() do teste de throughput
     PERF_FILE],
    capture_output=True, text=True, timeout=600, cwd=REPO_DIR,
)
elapsed_perf = time.perf_counter() - t0

stdout_perf = result_perf.stdout + result_perf.stderr

m_pass = re.search(r"(\d+) passed",  stdout_perf)
m_fail = re.search(r"(\d+) failed",  stdout_perf)
n_passed_perf = int(m_pass.group(1)) if m_pass else 0
n_failed_perf = int(m_fail.group(1)) if m_fail else 0

PERF_GATE_OK = n_failed_perf == 0 and result_perf.returncode == 0

# Extrair throughput de mod/h se disponível no stdout
throughputs = re.findall(r"([\d,.]+)\s*mod/h", stdout_perf)
if throughputs:
    print("Throughputs encontrados:")
    for th in throughputs:
        print(f"  {th} mod/h")

print(stdout_perf[-4000:])
print(f"\n{'[GATE PASS]' if PERF_GATE_OK else '[GATE FAIL]'} "
      f"Gate T1.6: {n_passed_perf} PASSED, {n_failed_perf} FAILED  ({elapsed_perf:.1f}s)")


## § 8 Export de Resultados para Google Drive

Salva um JSON estruturado com todos os resultados dos gates para
rastreabilidade e comparação entre execuções.

In [ ]:
# ── Célula 17: Export JSON → Drive ──────────────────────────────────────────
import json, datetime, os

# Informações do hardware
try:
    import jax
    device_info = str(jax.devices()[0])
except Exception:
    device_info = "unknown"

report = {
    "sprint":           "O1",
    "branch":           GIT_BRANCH,
    "git_tag":          GIT_TAG,
    "timestamp_utc":    datetime.datetime.utcnow().isoformat(),
    "hardware":         device_info,
    "jax_version":      getattr(jax, "__version__", "unknown"),
    "is_gpu":           IS_GPU,
    "gates": {
        "gpu_suite":         {"passed": total_passed,       "failed": total_failed,       "ok": GPU_GATE_OK},
        "fortran_parity":    {"passed": n_passed,           "failed": n_failed,           "ok": PARITY_GATE_OK},
        "perf_baseline_t16": {"passed": n_passed_perf,      "failed": n_failed_perf,      "ok": PERF_GATE_OK},
    },
    "gpu_test_files": [
        {
            "file":      r["file"],
            "passed":    r.get("passed",  0),
            "failed":    r.get("failed",  0),
            "skipped":   r.get("skipped", 0),
            "elapsed_s": round(r.get("elapsed_s", 0), 2),
            "status":    r.get("status",  "?"),
        }
        for r in GPU_RESULTS
    ],
    "o1_checks": [
        {"label": label, "ok": ok, "detail": str(detail)}
        for label, ok, detail in checks
    ],
    "total_time_s": round(total_time + elapsed_parity + elapsed_perf, 1),
}

# Salvar local sempre
ts = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
local_path = f"/tmp/sprint_o1_gpu_results_{ts}.json"
with open(local_path, "w") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)
print(f"[OK] Salvo localmente: {local_path}")

# Salvar no Drive se disponível
if IN_COLAB and os.path.exists(DRIVE_OUT):
    drive_path = os.path.join(DRIVE_OUT, f"sprint_o1_gpu_results_{ts}.json")
    import shutil
    shutil.copy(local_path, drive_path)
    print(f"[OK] Salvo no Drive: {drive_path}")
else:
    print(f"[INFO] Drive não disponível — apenas salvo em {local_path}")

# Preview do relatório
print("\n--- Sumário do Relatório ---")
print(json.dumps(report["gates"], indent=2))


## § 9 Decisão Go / No-Go para Merge em `main`

Consolida os três gates e emite a decisão final.

In [ ]:
# ── Célula 19: Decisão Go / No-Go ────────────────────────────────────────────
import datetime

print("=" * 70)
print("SPRINT O1 — DECISÃO FINAL")
print("=" * 70)
print(f"Branch : {GIT_BRANCH}")
print(f"Data   : {datetime.datetime.utcnow().strftime('%Y-%m-%d %H:%M UTC')}")
print(f"Hardware: {device_info}")
print()

gates = [
    ("Gate 1: Suite GPU (0 FAILED)",       GPU_GATE_OK,    f"{total_passed}P / {total_failed}F"),
    ("Gate 2: Paridade Fortran <1e-12",    PARITY_GATE_OK, f"{n_passed}P / {n_failed}F"),
    ("Gate 3: Performance T1.6",           PERF_GATE_OK,   f"{n_passed_perf}P / {n_failed_perf}F"),
]

all_pass = all(ok for _, ok, _ in gates)

for label, ok, detail in gates:
    status = "[PASS]" if ok else "[FAIL]"
    print(f"{status} {label:45s} ({detail})")

print()
print("-" * 70)
if all_pass:
    print("DECISÃO: ✅ GO — Merge feat/sprint-o1-quick-wins → main APROVADO")
    print()
    print("Próximos passos:")
    print("  1. git checkout main && git merge --no-ff feat/sprint-o1-quick-wins")
    print("  2. git tag -a v2.43 -m 'Sprint O1: JAX GPU Quick Wins'")
    print("  3. git push origin main --tags")
    print("  4. Iniciar Sprint O2: lax.map(batch_size=K) + async dispatch")
else:
    print("DECISÃO: ❌ NO-GO — Corrigir FAIL acima antes do merge")
    print()
    failed_gates = [(l, d) for l, ok, d in gates if not ok]
    for label, detail in failed_gates:
        print(f"  ► {label}: {detail}")
    print()
    print("Para depurar, rode a célula de pytest do gate falho com -v --tb=long")

print("-" * 70)
print(f"Tempo total: {report['total_time_s']:.0f}s")


## Apêndice — Comandos de Depuração

Se algum gate falhar, use as células abaixo para depurar individualmente.
Execute apenas as células relevantes (não fazem parte do fluxo principal).

In [ ]:
# ── Apêndice A: Rodar teste individual com --tb=long ─────────────────────────
# Edite TEST_TO_DEBUG e FUNC_TO_DEBUG conforme necessário
import subprocess, sys, os

TEST_TO_DEBUG = "tests/test_simulation_jax_sprint12.py"  # edite aqui
FUNC_TO_DEBUG = None  # ou "test_t5_vmap_real_paridade_oklahoma_3" para rodar 1 função

cmd = [
    sys.executable, "-m", "pytest",
    "-m", "gpu",
    "-v", "--tb=long", "-s",
]
if FUNC_TO_DEBUG:
    cmd.append(f"{os.path.join(REPO_DIR, TEST_TO_DEBUG)}::{FUNC_TO_DEBUG}")
else:
    cmd.append(os.path.join(REPO_DIR, TEST_TO_DEBUG))

result = subprocess.run(cmd, capture_output=True, text=True, timeout=300, cwd=REPO_DIR)
print(result.stdout[-6000:])
if result.stderr:
    print("STDERR:", result.stderr[-1000:])


In [ ]:
# ── Apêndice B: Inspecionar cache XLA ────────────────────────────────────────
import os, glob

cache_dir = os.environ.get("JAX_COMPILATION_CACHE_DIR", "/tmp/jax_compilation_cache_geosteering")
print(f"Cache dir: {cache_dir}")

if os.path.exists(cache_dir):
    files = list(glob.glob(os.path.join(cache_dir, "**", "*"), recursive=True))
    total_bytes = sum(os.path.getsize(f) for f in files if os.path.isfile(f))
    print(f"Arquivos: {len(files)}")
    print(f"Tamanho total: {total_bytes / 1024 / 1024:.1f} MB")
    # Lista os maiores
    file_sizes = [(os.path.getsize(f), f) for f in files if os.path.isfile(f)]
    file_sizes.sort(reverse=True)
    print("\nMaiores arquivos:")
    for sz, path in file_sizes[:10]:
        print(f"  {sz/1024:.0f} KB  {os.path.basename(path)}")
else:
    print("Cache dir não existe — XLA cache não foi populado ainda")


In [ ]:
# ── Apêndice C: Inspecionar LRU cache Hankel ─────────────────────────────────
import sys
sys.path.insert(0, REPO_DIR)

from geosteering_ai.simulation._jax.kernel import get_hankel_filter_jax

# 1ª chamada (preenche cache)
kr, wJ0, wJ1 = get_hankel_filter_jax("werthmuller_201pt")
print(f"werthmuller_201pt: kr.shape={kr.shape}, dtype={kr.dtype}")
print(f"wJ0.shape={wJ0.shape}, wJ1.shape={wJ1.shape}")
print(f"wJ0 device: {wJ0.devices()}")

# Info do lru_cache (se acessível)
# get_hankel_filter_jax é wrapping de _get_hankel_filter_cached
try:
    from geosteering_ai.simulation._jax.kernel import _get_hankel_filter_cached
    info = _get_hankel_filter_cached.cache_info()
    print(f"\nlru_cache info: hits={info.hits}, misses={info.misses}, maxsize={info.maxsize}")
except (ImportError, AttributeError) as e:
    print(f"lru_cache info não disponível: {e}")
